In [ ]:
# ============================================
# GOLD LAYER - BUSINESS METRICS
# ============================================

import logging
from pyspark.sql.functions import col, sum, month, year, when, count

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# LOAD
sales = spark.read.table("silver_sales_clean")
exchange = spark.read.table("silver_exchange_rate_clean")

# PREPARE
sales = sales.withColumn("month", month(col("order_date"))) \
             .withColumn("year", year(col("order_date")))

# JOIN
df = sales.join(exchange, ["month", "year"], "left")

df = df.withColumn(
    "exchange_rate",
    when(col("exchange_rate").isNull(), 1).otherwise(col("exchange_rate"))
)

# KPI 1
df = df.withColumn(
    "revenue_vnd",
    col("final_amount") * col("exchange_rate")
)

revenue = df.groupBy("year", "month", "items") \
    .agg(sum("revenue_vnd").alias("total_revenue_vnd"))

# KPI 2
promo = sales.withColumn(
    "is_discount",
    when(col("discount_code").isNotNull(), 1).otherwise(0)
)

promo_result = promo.groupBy("location") \
    .agg((sum("is_discount") / count("*") * 100).alias("discount_percentage"))

# WRITE
revenue.write.mode("overwrite").saveAsTable("gold_revenue_summary")
promo_result.write.mode("overwrite").saveAsTable("gold_promotion_effectiveness")

print("Sample revenue:")
spark.read.table("gold_revenue_summary").limit(5).show()

mssparkutils.notebook.exit("Success")